In [2]:
#%pip install requests
#%pip install telethon
#%pip install --upgrade ipykernel
#%pip install requests lxml pandas
#%pip install selenium
#%pip install pymongo

In [3]:
#%pip install requests lxml mysql-connector-python

парсинг сайта с помощью scrapy


In [4]:
#%pip install crochet
#%pip install scrapy
import scrapy
from scrapy.crawler import CrawlerProcess
import crochet

In [5]:
from pathlib import Path

# БАЗА: папка, где лежит ноутбук (переносимо)
BASE_PATH    = Path(".")
PROJECT_ROOT = BASE_PATH / "simple_scrapy_spider"                  # тут scrapy.cfg
SPIDER_DIR   = PROJECT_ROOT / "simple_scrapy_spider" / "spiders"   # тут пауки
SPIDER_FILE  = SPIDER_DIR / "oldgames_catalog.py"
OUTPUT_CSV   = BASE_PATH / "oldgames_dos.csv"

print("BASE_PATH   :", BASE_PATH.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("SPIDER_DIR  :", SPIDER_DIR.resolve())
print("SPIDER_FILE :", SPIDER_FILE.resolve())
print("OUTPUT_CSV  :", OUTPUT_CSV.resolve())

# sanity-check
assert (PROJECT_ROOT / "scrapy.cfg").is_file(), "Не найден scrapy.cfg в папке проекта (simple_scrapy_spider)."

BASE_PATH   : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2
PROJECT_ROOT: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider
SPIDER_DIR  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders
SPIDER_FILE : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\oldgames_catalog.py
OUTPUT_CSV  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv


In [ ]:
SPIDER_DIR.mkdir(parents=True, exist_ok=True)

fixed_spider = r'''
import re
import scrapy

class OldGamesSpider(scrapy.Spider):
    name = "oldgames"
    allowed_domains = ["old-games.ru", "www.old-games.ru", "static.old-games.ru"]

    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 0.5,  # 
        "DEFAULT_REQUEST_HEADERS": {
            "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36 (Scrapy for research/edu)",
        },
        "FEED_EXPORT_ENCODING": "utf-8-sig",
    }

    def __init__(self, max_pages=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.page_counter = 0
        self.max_pages = int(max_pages) if max_pages else None
        self.start_urls = [
            "https://www.old-games.ru/catalog/?platform=1&sort=popularity"
        ]

    def parse(self, response):
        self.page_counter += 1

        # ВАЖНО: используем ТОЛЬКО прямые дети ./td[n] (первый столбец содержит вложенную таблицу)
        rows = response.xpath('//table[contains(@class,"listtable")]/tr[starts-with(@id,"game_")]')
        for row in rows:
            # 1) Название и ссылка
            name_link = row.xpath(
                './td[1]//a[starts-with(@href,"/game/") and '
                'not(contains(@href,"/video/")) and '
                'not(contains(@href,"/screenshots/")) and '
                'not(contains(@href,"/covers/"))][1]'
            )
            name = name_link.xpath('normalize-space(text())').get()
            url  = response.urljoin(name_link.xpath('@href').get())

            # 2) Жанр (может быть несколько)
            genres = row.xpath('./td[2]//a/text()').getall()
            genre = " / ".join([g.strip() for g in genres if g.strip()]) or None

            # 3) Год
            year_text = row.xpath('./td[3]//a/text()').re_first(r'\d{4}')
            year = int(year_text) if year_text else None

            # 4) Платформа (оставим fallback на текст)
            platforms = row.xpath('./td[4]//a/text()').getall()
            platform = " / ".join([p.strip() for p in platforms if p.strip()])
            if not platform:
                platform = row.xpath('normalize-space(./td[4])').get() or None

            # 5) Издатель 
            publishers = row.xpath('./td[5]//a/text()').getall()
            publisher = " / ".join([p.strip() for p in publishers if p.strip()])
            if not publisher:
                publisher = row.xpath('normalize-space(./td[5])').get() or None

            # 6) Оценка: число из title у <img> «… — N из 10»
            rating_title = row.xpath('./td[6]//img/@title').get()
            rating = None
            if rating_title:
                m = re.search(r'(\d+)\s*из\s*10', rating_title)
                if m:
                    rating = int(m.group(1))

            yield {
                "Название":  name or None,
                "Жанр":      genre or None,
                "Год":       year,
                "Платформа": platform or None,
                "Издатель":  publisher or None,
                "Оценка":    rating,
                "Ссылка": url,  
            }

        # Пагинация (rel="next" или кнопка ">")
        if self.max_pages is None or self.page_counter < self.max_pages:
            next_rel = response.xpath(
                '//ul[contains(@class,"pager")]//a[@rel="next"]/@href | '
                '//ul[contains(@class,"pager")]//a[normalize-space(text())=">"]/@href'
            ).get()
            if next_rel:
                yield response.follow(next_rel, callback=self.parse)
'''

SPIDER_FILE.write_text(fixed_spider.strip() + "\n", encoding="utf-8")
print("Паук обновлён:", SPIDER_FILE)


Паук обновлён: simple_scrapy_spider\simple_scrapy_spider\spiders\oldgames_catalog.py


In [7]:
import sys, subprocess

cmd = [
    sys.executable, "-m", "scrapy", "crawl", "oldgames",
    "-O", str(OUTPUT_CSV.resolve()),  # абсолютный путь, собранный из относительного
    "-a", "max_pages=2",              # для теста: соберёт 2 страницы; если убрать параметр — пойдёт по всем
    "-s", "LOG_LEVEL=INFO",
]
print("Запуск:", " ".join(cmd))
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT.resolve()))
print("Exit code:", res.returncode)
print("CSV:", OUTPUT_CSV.resolve(), "=>", OUTPUT_CSV.exists())

Запуск: c:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\.venv\Scripts\python.exe -m scrapy crawl oldgames -O C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv -a max_pages=2 -s LOG_LEVEL=INFO
Exit code: 0
CSV: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv => True


In [8]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
display(df.head(20))

print("\nПроверки на пустые значения:")
for col in ["Жанр", "Платформа", "Издатель"]:
    print(f"{col}: пустых = {int(df[col].isna().sum())}")

,Название,Жанр,Год,Платформа,Издатель,Оценка,Ссылка
0,WarCraft II: Tides of Darkness,Strategy,1995,DOS,Blizzard Entertainment,10,https://www.old-games.ru/game/73.html
1,Blood,Action,1997,DOS,GT Interactive Software,10,https://www.old-games.ru/game/11.html
2,Quake,Action,1996,DOS,id Software,10,https://www.old-games.ru/game/64.html
3,X-COM: UFO Defense,Strategy,1994,DOS,MicroProse Software,8,https://www.old-games.ru/game/77.html
4,DOOM,Action,1993,DOS,id Software,10,https://www.old-games.ru/game/4788.html
5,Dune II: The Building of a Dynasty,Strategy,1992,DOS,Virgin Games,10,https://www.old-games.ru/game/1482.html
6,Wolfenstein 3D,Action,1992,DOS,Apogee Software,8,https://www.old-games.ru/game/76.html
7,Sid Meier's Civilization,Strategy,1991,DOS,MicroProse Software,10,https://www.old-games.ru/game/12.html
8,Duke Nukem 3D: Atomic Edition,Action,1996,DOS,GT Interactive Software,10,https://www.old-games.ru/game/604.html
9,Prince of Persia,Arcade,1990,DOS,Brøderbund Software,10,https://www.old-games.ru/game/62.html



Проверки на пустые значения:
Жанр: пустых = 0
Платформа: пустых = 0
Издатель: пустых = 0


In [1]:
from pathlib import Path

BASE_PATH    = Path(".")
PROJECT_ROOT = BASE_PATH / "simple_scrapy_spider"
SPIDER_DIR   = PROJECT_ROOT / "simple_scrapy_spider" / "spiders"

SPIDER_FILE  = SPIDER_DIR / "olx_transport.py"

MASTER_CSV   = BASE_PATH / "olx_transport_master.csv"   # “накопительный” файл
RUN_CSV      = BASE_PATH / "olx_transport_run.csv"      # файл текущего запуска (временный)

SPIDER_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("SPIDER_FILE :", SPIDER_FILE.resolve())
print("MASTER_CSV  :", MASTER_CSV.resolve())
print("RUN_CSV     :", RUN_CSV.resolve())

assert (PROJECT_ROOT / "scrapy.cfg").is_file(), "Не найден scrapy.cfg в папке проекта (simple_scrapy_spider)."


PROJECT_ROOT: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider
SPIDER_FILE : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\olx_transport.py
MASTER_CSV  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_master.csv
RUN_CSV     : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_run.csv


In [ ]:
fixed_spider = r'''
import re
import scrapy


def _norm_spaces(s: str | None) -> str | None:
    if not s:
        return None
    return re.sub(r"\s+", " ", s).strip()


def _re_first(text: str, pattern: str) -> str | None:
    m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
    return m.group(1).strip() if m else None


def _kv(text: str, labels: list[str]) -> str | None:
    # Ищем "Лейбл: значение" в тексте страницы
    for lbl in labels:
        m = re.search(rf"^{re.escape(lbl)}\s*:\s*(.+)$", text, flags=re.IGNORECASE | re.MULTILINE)
        if m:
            return m.group(1).strip()
    return None


def _guess_model_from_title(title: str | None) -> str | None:
    # очень грубый fallback: убираем год (4 цифры) и хвост после него
    if not title:
        return None
    t = _norm_spaces(title)
    m = re.search(r"^(.*?)(?:\b(19|20)\d{2}\b.*)?$", t)
    base = m.group(1).strip() if m else t
    return base or t


class OlxTransportSpider(scrapy.Spider):
    name = "olx_transport"
    allowed_domains = ["olx.ua", "www.olx.ua"]

    custom_settings = {
        # Важно: если ROBOTSTXT_OBEY=True и robots запрещает — паук может собрать 0 строк.
        
        "ROBOTSTXT_OBEY": False,

        "DOWNLOAD_DELAY": 0.7,
        "AUTOTHROTTLE_ENABLED": True,
        "FEED_EXPORT_ENCODING": "utf-8-sig",
        "DEFAULT_REQUEST_HEADERS": {
            "Accept-Language": "uk-UA,uk;q=0.9,ru-RU;q=0.8,ru;q=0.7,en-US;q=0.6,en;q=0.5",
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
        },
    }

    def __init__(self, start_url=None, max_pages=1, max_items=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.start_urls = [start_url or "https://www.olx.ua/uk/transport/"]
        self.max_pages = int(max_pages) if max_pages else None
        self.max_items = int(max_items) if max_items else None
        self.page_counter = 0
        self.item_counter = 0

    def parse(self, response):
        self.page_counter += 1

        cards = response.xpath('//div[@data-cy="l-card"]')
        for card in cards:
            if self.max_items and self.item_counter >= self.max_items:
                break

            title = _norm_spaces(card.xpath('.//*[@data-testid="ad-card-title"]/text()').get())
            price = _norm_spaces(card.xpath('.//*[@data-testid="ad-price"]/text()').get())

            href = card.xpath('.//a[contains(@href,"/d/")]/@href').get()
            if not href:
                continue
            url = response.urljoin(href.split("?")[0])

            # Короткие характеристики часто “спрятаны” в span с data-testid вида "*card-param*"
            short_params = card.xpath('.//span[contains(@data-testid,"card-param")]/text()').getall()
            short_params = ", ".join([_norm_spaces(x) for x in short_params if _norm_spaces(x)]) or None

            self.item_counter += 1
            yield response.follow(
                url,
                callback=self.parse_offer,
                meta={
                    "listing_title": title,
                    "listing_price": price,
                    "listing_short": short_params,
                    "listing_url": url,
                },
            )

        # пагинация (на OLX есть кнопка вперёд)
        if self.max_pages is None or self.page_counter < self.max_pages:
            next_href = response.xpath('//a[@data-testid="pagination-forward"]/@href').get()
            if next_href:
                yield response.follow(next_href, callback=self.parse)

    def parse_offer(self, response):
        # Берём много “человеческого” текста страницы, чтобы доставать "Лейбл: Значение"
        texts = [t.strip() for t in response.xpath('//text()').getall() if t.strip()]
        joined = "\n".join(texts)

        # Название
        title = _norm_spaces(response.xpath('normalize-space(//h1[1])').get()) or response.meta.get("listing_title")

        # ID (на OLX обычно есть строка вида "ID: 123456789")
        ad_id = _re_first(joined, r"\bID:\s*(\d+)\b")

        # Цена (fallback на то, что было на карточке)
        price = _re_first(joined, r"^###\s*([0-9\s]+(?:грн\.|UAH|₴).*)$")  # иногда в тексте встречается как заголовок
        price = _norm_spaces(price) or response.meta.get("listing_price")

        # Модель: пытаемся собрать "Марка + Модель" из параметров
        brand = _kv(joined, ["Марка", "Бренд"])
        model_only = _kv(joined, ["Модель"])
        model = " ".join([x for x in [brand, model_only] if x]) or _guess_model_from_title(title)

        # Состояние: "Стан" (укр) или "Состояние" (рус)
        condition = _kv(joined, ["Стан", "Состояние", "Стан авто"])

        # Краткие характеристики: собираем несколько “типовых” полей
        char_labels = [
            "Рік випуску", "Год выпуска",
            "Пробіг", "Пробег",
            "Тип кузова", "Тип палива", "Вид палива",
            "Коробка передач",
            "Об'єм двигуна", "Объем двигателя",
            "Кількість дверей", "Количество дверей",
            "Колір", "Цвет",
        ]
        parts = []
        for lbl in char_labels:
            v = _kv(joined, [lbl])
            if v:
                parts.append(f"{lbl}: {v}")
        characteristics = "; ".join(parts) if parts else response.meta.get("listing_short")

        yield {
            "ID": ad_id,
            "Название": title,
            "Модель": model,
            "Краткие характеристики": characteristics,
            "Состояние": condition,
            "Цена": price,
            "Ссылка": response.meta.get("listing_url") or response.url,
        }
'''

SPIDER_FILE.write_text(fixed_spider.strip() + "\n", encoding="utf-8")
print("Паук сохранён:", SPIDER_FILE)


Паук сохранён: simple_scrapy_spider\simple_scrapy_spider\spiders\olx_transport.py


In [3]:
import sys, subprocess

cmd = [
    sys.executable, "-m", "scrapy", "crawl", "olx_transport",
    "-O", str(RUN_CSV.resolve()),      # перезаписываем ТОЛЬКО временный файл запуска
    "-a", "max_pages=1",             
    # "-a", "start_url=https://www.olx.ua/uk/transport/legkovye-avtomobili/",  
    "-s", "LOG_LEVEL=INFO",
]

print("Запуск:", " ".join(cmd))
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT.resolve()))
print("Exit code:", res.returncode)
print("RUN_CSV exists:", RUN_CSV.exists(), "=>", RUN_CSV.resolve())


Запуск: c:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\.venv\Scripts\python.exe -m scrapy crawl olx_transport -O C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_run.csv -a max_pages=1 -s LOG_LEVEL=INFO
Exit code: 0
RUN_CSV exists: True => C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_run.csv


In [5]:
import pandas as pd

new_df = pd.read_csv(RUN_CSV, encoding="utf-8-sig")

if MASTER_CSV.exists():
    old_df = pd.read_csv(MASTER_CSV, encoding="utf-8-sig")
    df = pd.concat([old_df, new_df], ignore_index=True)
else:
    df = new_df.copy()

# Дедупликация: сначала по ID, если пусто — по ссылке
key = df["ID"].fillna(df["Ссылка"]) if "ID" in df.columns else df["Ссылка"]
df["__key"] = key

df = df.drop_duplicates(subset="__key", keep="first").drop(columns="__key")

df.to_csv(MASTER_CSV, index=False, encoding="utf-8-sig")

display(df.head(30))
print("Итоговых строк:", len(df))
print("MASTER_CSV:", MASTER_CSV.resolve())


,ID,Название,Модель,Краткие характеристики,Состояние,Цена,Ссылка
0,870576653,NaN,NaN,Рік випуску: 2007,З пробігом,358 632 грн.,https://www.olx.ua/d/uk/obyavlenie/konteynerov...
1,908021831,NaN,NaN,Рік випуску: 2025; Пробіг: 0 км,Новий,75 000 грн.,https://www.olx.ua/d/uk/obyavlenie/pritsep-1pt...
2,908882237,NaN,Octavia,Рік випуску: 2007; Пробіг: 300 тис.км.; Тип ку...,NaN,88 603 грн.,https://www.olx.ua/d/uk/obyavlenie/skoda-oktav...
3,909502836,NaN,Corsa,Рік випуску: 2008; Пробіг: 147 тис.км.; Тип ку...,NaN,208 850 грн.,https://www.olx.ua/d/uk/obyavlenie/opel-corsa-...
4,908087619,NaN,100,Рік випуску: 1991; Пробіг: 350 тис.км.; Тип ку...,NaN,118 137 грн.,https://www.olx.ua/d/uk/obyavlenie/prodam-aud-...
5,837119994,NaN,NaN,Рік випуску: 2025; Пробіг: 5 км; Коробка перед...,Новий,63 000 грн.,https://www.olx.ua/d/uk/obyavlenie/vagi-na-fro...
6,909790254,NaN,Sportage,Рік випуску: 2006; Пробіг: 329 тис.км.; Тип ку...,NaN,202 521 грн.,https://www.olx.ua/d/uk/obyavlenie/ka-sportege...
7,906404557,NaN,NaN,Рік випуску: 2025,Новий,242 000 грн.,https://www.olx.ua/d/uk/obyavlenie/elvorti-ast...
8,909185071,NaN,NaN,Рік випуску: 1995; Пробіг: 2 км,З пробігом,274 205 грн.,https://www.olx.ua/d/uk/obyavlenie/kemper-kara...
9,908601489,NaN,5 серія,Рік випуску: 2001; Пробіг: 300 тис.км.; Тип ку...,NaN,181 425 грн.,https://www.olx.ua/d/uk/obyavlenie/prodam-bmw-...


Итоговых строк: 52
MASTER_CSV: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_master.csv
